# 3. Translation

Takes the long files built by `Compendium_1_Long_Files.ipynb` and produces one
English file per chapter:

1. **Translate** `merged_long_files\<Chapter>_AR.xlsx` into English.
2. **Append** `merged_long_files\<Chapter>_EN.xlsx` if that chapter also had
   English questionnaires - those rows are already English and need no
   translation.
3. **Save** the combined result as `COMPENDIUM-ARAB SOCIETY\<Chapter>_EN.xlsx`.

The Arabic file already contains the calculated indicators - notebook 2 added
them before this runs - so they are translated along with everything else and
never need a round trip back into Arabic.

The combined file is written to a third location and rebuilt from scratch each
run, so `merged_long_files` stays purely what notebook 1 put there and
re-running can never append the same rows twice.

Anything the dictionary has no entry for passes through untranslated and is
listed at the end, ready for `export_untranslated()` -> Claude Code ->
`update_dictionary()`.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config / paths


In [ ]:
"""
CELL: Configuration - paths and chapters.
"""
DATA_COLLECTOR_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "DATA COLLECTOR"
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
COMPENDIUM_PATH = Path.home() / "OneDrive - United Nations" / "Desktop" / "DSS" / "COMPENDIUM-ARAB SOCIETY"

# Every long file lives here - one folder, both languages. The _AR / _EN suffix
# is already in each filename, so a folder per language only meant two places to
# look.
LONG_FILES_PATH = COMPENDIUM_PATH / "merged_long_files"

# Questionnaire folders are named <prefix><LANGUAGE>. The suffix IS the language,
# so nothing here has to list them - add a folder and it is picked up.
QUESTIONNAIRE_PREFIX = "datacollector_received_quest_"

# One long file per chapter per language lands here.


# Leave CHAPTERS as None to process every chapter found on disk. Set an explicit
# list to restrict one run, e.g. CHAPTERS = ["Poverty"].
CHAPTERS = None

LANGUAGES = ["AR", "EN"]

# The language the questionnaires mostly arrive in, and the one we translate to.
SOURCE_LANGUAGE = "AR"
TARGET_LANGUAGE = "EN"


def discover_chapters():
    """Chapters that notebook 1 produced a long file for, in either language."""
    names = set()
    for language in LANGUAGES:
        folder = LONG_FILES_PATH
        if not folder.exists():
            continue
        suffix = f"_{language}.xlsx"
        for path in folder.glob(f"*{suffix}"):
            names.add(path.name[: -len(suffix)])
    return sorted(names)


def chapters_to_process():
    """CHAPTERS if it was set, otherwise whatever notebook 1 produced."""
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    logger.info(f"Chapters with a long file: {found}")
    if not found:
        logger.warning("No long files found - run notebook 1 first.")
    return found


# ---------------------------------------------------------------------------
# Everything this run fixed, inferred, or added to the dictionary on its own
# goes to pipeline_changes_<Chapter>.txt; everything it left alone, and why,
# goes to need manual intervention_<Chapter>.txt - one pair of files per
# chapter, plus a _general pair for a dictionary-level finding with no
# chapter of its own. Each notebook owns a section and rewrites only its
# own, so either file always reflects the latest run of each step whatever
# order they ran in. See save_inconsistencies() below.


def changes_log_path(chapter):
    """Where one chapter's own auto-applied changes live - what this run
    corrected, inferred, or added to the dictionary on its own."""
    return COMPENDIUM_PATH / f"pipeline_changes_{chapter}.txt"


def manual_intervention_log_path(chapter):
    """Where one chapter's own open findings live - what this run left
    alone, and why, for a person to look at."""
    return COMPENDIUM_PATH / f"need manual intervention_{chapter}.txt"


GENERAL_CHANGES_LOG_PATH = COMPENDIUM_PATH / "pipeline_changes_general.txt"
GENERAL_MANUAL_LOG_PATH = COMPENDIUM_PATH / "need manual intervention_general.txt"

# A record's own `kind` decides which of the two files it lands in - fixed,
# inferred, or added to the dictionary goes to pipeline_changes_<Chapter>.txt;
# everything else (a gap, a skip, a collision) goes to
# need manual intervention_<Chapter>.txt. Anything not listed here defaults
# to manual intervention, on purpose - a new kind nobody has classified yet
# should surface for a person to see, not disappear into "already handled".
CHANGE_KINDS = {
    "column name corrected",
    "value corrected",
    "value needed correcting",
    "reused a close Arabic spelling",
    "dictionary entry added",
}


def _write_section(path, title, section, section_text):
    """Replace one named section in one file, in place, leaving every other
    section exactly as it was. The mechanic every chapter file and the
    general one share - split what's there into sections by name, replace
    this one, rebuild in section order so the file reads the same whatever
    order the notebooks last ran in."""
    header = [title, "=" * 78, ""]

    sections = {}
    if path.exists():
        existing = path.read_text(encoding="utf-8")
        parts = re.split(r"^### (.+?) ###$", existing, flags=re.M)
        for name, text in zip(parts[1::2], parts[2::2]):
            sections[name] = f"### {name} ###{text.rstrip()}"
    sections[section] = section_text

    body_text = "\n\n".join(sections[name] for name in sorted(sections))
    path.write_text("\n".join(header).rstrip("\n") + "\n\n" + body_text + "\n",
                    encoding="utf-8")


def save_inconsistencies(section, records, chapters=None):
    """Write this notebook's findings for this section, split into two files
    per chapter - pipeline_changes_<Chapter>.txt for everything this run
    fixed, inferred, or added to the dictionary on its own, and
    need manual intervention_<Chapter>.txt for everything it left alone and
    why. A finding with no chapter (a dictionary-level problem, not a
    source-data one) goes to the matching _general file instead.

    `chapters` is every chapter this call actually covers, independent of
    whether any of them have a finding - pass it explicitly so a clean
    chapter still gets both files correctly replaced with "Nothing found"
    rather than left showing whatever an earlier, unrelated run left there.
    """
    stamp = pd.Timestamp.now().strftime("%d %B %Y, %H:%M")
    marker = f"### {section} ###"

    WHERE = ["country", "indicator", "year", "sex", "age_group",
             "nationality", "area", "file", "sheet", "row"]

    def render(chapter_records):
        body = [marker, f"    last run {stamp}", ""]
        if not chapter_records:
            body += ["    Nothing found.", ""]
        else:
            frame = pd.DataFrame(chapter_records)
            for kind, group in frame.groupby("kind", sort=False):
                body.append(f"  {kind.upper()}  ({len(group)})")
                for _, row in group.iterrows():
                    def show(value):
                        if isinstance(value, float) and float(value).is_integer():
                            return str(int(value))
                        return str(value)
                    where = " · ".join(
                        show(row[f]) for f in WHERE
                        if f in row and pd.notna(row[f]) and str(row[f]) != "")
                    body.append(f"      {where}" if where else "      -")
                    body.append(f"          {row['detail']}")
                body.append("")
        return "\n".join(body)

    def split(chapter_records):
        changes = [r for r in chapter_records if r["kind"] in CHANGE_KINDS]
        manual = [r for r in chapter_records if r["kind"] not in CHANGE_KINDS]
        return changes, manual

    by_chapter = defaultdict(list)
    general = []
    for record in records:
        chapter = record.get("chapter")
        if chapter in (None, "", "-"):
            general.append(record)
        else:
            by_chapter[str(chapter)].append(record)

    covered = {str(c) for c in (chapters or [])} | set(by_chapter)

    paths = []
    for chapter in sorted(covered):
        changes, manual = split(by_chapter.get(chapter, []))
        _write_section(changes_log_path(chapter), f"PIPELINE CHANGES — {chapter}",
                       section, render(changes))
        _write_section(manual_intervention_log_path(chapter),
                       f"NEED MANUAL INTERVENTION — {chapter}", section, render(manual))
        paths += [changes_log_path(chapter), manual_intervention_log_path(chapter)]

    if general or not covered:
        changes, manual = split(general)
        _write_section(GENERAL_CHANGES_LOG_PATH, "PIPELINE CHANGES — general",
                       section, render(changes))
        _write_section(GENERAL_MANUAL_LOG_PATH, "NEED MANUAL INTERVENTION — general",
                       section, render(manual))
        paths += [GENERAL_CHANGES_LOG_PATH, GENERAL_MANUAL_LOG_PATH]

    return paths, len(records)


## Load the translation dictionary

One dictionary, Arabic to English - no reverse is built, because the
questionnaires arrive in Arabic and English is what we translate into.


In [ ]:
"""
CELL: Load translation dict.xlsx - one dictionary, Arabic to English.
"""


def load_dictionary():
    """Reads translation dict.xlsx into:

      DICTIONARY_AR_TO_EN  (column_map, value_map)
          column_map = {Arabic column name: English column name}
          value_map  = {Arabic column name: {Arabic value: English value}}

      ENGLISH_VOCABULARY  (column_names, values_by_column)
          the English column names and values the file knows about.

    Only the Arabic-to-English direction is built, because the questionnaires
    arrive in Arabic and English is what we translate into.

    ENGLISH_VOCABULARY is not a reverse dictionary - it maps nothing. It is just
    the list of correct English spellings, so an English questionnaire's
    misspelled labels can be fuzzy-matched against the right words too.
    """
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map, value_map = {}, {}
    for arabic_column in dict_df["col_ar"].dropna().unique():
        rows = dict_df[dict_df["col_ar"] == arabic_column]
        column_map[arabic_column] = rows["col_en"].iloc[0]
        value_map[arabic_column] = {
            arabic: english
            for arabic, english in zip(rows["val_ar"], rows["val_en"])
            if pd.notna(arabic)
        }

    english_columns = {}
    english_values = {}
    for english_column in dict_df["col_en"].dropna().unique():
        rows = dict_df[dict_df["col_en"] == english_column]
        english_columns[english_column] = english_column
        english_values[english_column] = {
            str(v): str(v) for v in rows["val_en"].dropna().unique()
        }

    chapter_rows = dict_df[dict_df["col_en"] == "Chapter"]
    chapter_to_arabic = dict(zip(chapter_rows["val_en"], chapter_rows["val_ar"]))

    return (column_map, value_map), (english_columns, english_values), chapter_to_arabic


DICTIONARY_AR_TO_EN, ENGLISH_VOCABULARY, CHAPTER_TO_ARABIC = load_dictionary()


def vocabulary(language):
    """The known column names and values for a language, as
    (column_names, values_by_column) - what misspellings are matched against."""
    return DICTIONARY_AR_TO_EN if language == "AR" else ENGLISH_VOCABULARY


def column_name(arabic_name, language):
    """One of the pipeline's own column names, spelled for the given language."""
    arabic_to_english, _ = DICTIONARY_AR_TO_EN
    return arabic_name if language == "AR" else arabic_to_english[arabic_name]


arabic_columns, _ = DICTIONARY_AR_TO_EN
logger.info(f"Dictionary loaded: {len(arabic_columns)} column names, Arabic -> English")


## `translate()`

Swaps every Arabic value for its English equivalent, then renames the column.
Anything the dictionary has no entry for is left exactly as it is.


In [ ]:
"""
CELL: translate() - Arabic to English, using the one dictionary.
"""


def translate(table):
    """Swaps every Arabic value for its English equivalent, then renames the
    column to its English name.

    Values are replaced first and the column renamed second, because the value
    lookup is keyed by the column's ORIGINAL Arabic name - renaming first would
    lose it. Anything the dictionary has no entry for is left exactly as it is,
    and is picked up afterwards by find_untranslated().
    """
    column_map, value_map = DICTIONARY_AR_TO_EN

    table = table.copy()
    for column in list(table.columns):
        if column in value_map:
            table[column] = table[column].replace(value_map[column])
        if column in column_map:
            table = table.rename(columns={column: column_map[column]})
    return table


def looks_arabic(text):
    """True if the text contains at least one Arabic letter. U+0600-U+06FF is
    the Arabic Unicode block; English text has nothing in it."""
    return any("\u0600" <= character <= "\u06ff" for character in str(text))


## Finding and filling the dictionary's gaps

### Letting Claude Code close the gaps for you

You do not have to shuttle the file back and forth. Ask once:

> **"run the pipeline and fill any dictionary gaps"**

Claude Code runs the notebook, translates whatever the dictionary did not know,
writes it back into `translation dict.xlsx` (taking a backup first), re-runs to
confirm the gap list is empty, and reports what it added. The protocol is
recorded in `CLAUDE.md`, so it does not need explaining again each session.

Doing it by hand is still the same three calls: `export_untranslated(REPORTS)`
-> fill in the blank column -> `update_dictionary(filled, chapter="<Chapter>")` -
passing `chapter` so the rows added are logged to that chapter's own
`pipeline_changes_<Chapter>.txt`, not the `_general` file.


In [ ]:
"""
CELL: find_untranslated() - what the dictionary could not translate.
"""


def find_untranslated(arabic_table, translated_table):
    """Values that came out of translate() unchanged and are still Arabic.

    A value with no dictionary entry is copied through untouched, so comparing
    the table before and against after finds them exactly. Values already in
    Latin script are not gaps - plenty of Arabic questionnaires cite their
    source in English ("MICS 2022", a URL) and those are correct as they stand.
    """
    column_map, _ = DICTIONARY_AR_TO_EN
    gaps = []

    for arabic_column, english_column in zip(arabic_table.columns, translated_table.columns):
        before = arabic_table[arabic_column]
        after = translated_table[english_column]
        pairs = pd.DataFrame({"val_ar": before, "val_en": after}).dropna()

        for (arabic_value, english_value), count in pairs.groupby(["val_ar", "val_en"]).size().items():
            if str(arabic_value).strip() != str(english_value).strip():
                continue                      # translated fine
            if not looks_arabic(arabic_value):
                continue                      # already English
            # Where does it appear? One example row is enough to find it.
            example = arabic_table[arabic_table[arabic_column] == arabic_value].head(1)
            context = {}
            for field, english_name in [("country", "Country"),
                                        ("indicator", "Indicator")]:
                arabic_name = {v: k for k, v in column_map.items()}.get(english_name)
                if arabic_name in arabic_table.columns and len(example):
                    context[field] = example.iloc[0][arabic_name]
            year_column = column_map and next(
                (a for a, e in column_map.items() if e == "Year"), None)
            if year_column in arabic_table.columns and len(example):
                context["year"] = example.iloc[0][year_column]

            gaps.append({
                "col_ar": arabic_column, "col_en": english_column,
                "val_ar": arabic_value, "val_en": None, "rows": count, **context,
            })
    return gaps


def gap_records(reports):
    """The dictionary gaps as records for the shared log.

    Each carries the country, indicator and year of an example row, so a missing
    term can be traced back to the questionnaire it came from rather than only
    being named.
    """
    records, seen = [], set()
    for report in reports:
        for gap in report["untranslated"]:
            key = (gap["col_ar"], gap["val_ar"])
            if key in seen:
                continue
            seen.add(key)
            record = {"kind": "no English translation", "chapter": report["chapter"],
                      "detail": f"{gap['col_en']}: {gap['val_ar']!r} appears in "
                                f"{gap['rows']} row(s) and passes through untranslated"}
            for field in ("country", "indicator", "year"):
                if gap.get(field) is not None:
                    record[field] = gap[field]
            records.append(record)
    return records


def dictionary_key(col_ar, val_ar):
    """The (column, value) pair that makes a dictionary row unique - with NaN
    normalized to None first. A float NaN never equals another float NaN
    (nan != nan is True), so leaving it as-is would make a column-defining
    row (val_ar genuinely blank, by design - see update_dictionary() below)
    look "new" every time, defeating the whole point of checking for it:
    called twice, it would add itself twice.
    """
    return (col_ar, val_ar if pd.notna(val_ar) else None)


DICTIONARY_LOG = []  # every row any update_dictionary() call has added this run


def update_dictionary(filled, chapter=None, backup=True):
    """Appends reviewed translations to translation dict.xlsx.

    `filled` is the exported table with val_en filled in (a DataFrame, or a path
    to the saved file). An Arabic value the dictionary already has is left
    alone - so running this twice changes nothing the second time. A
    timestamped backup is written first, because this edits the project's
    source of truth.

    A row may teach a COLUMN name instead of a value translation - val_ar and
    val_en both genuinely blank, because a "column not in the dictionary" fix
    has no value to pair it with. Those are kept; only a row that is neither a
    complete value pair nor a column-only row is skipped.

    `chapter` names the run this call belongs to, so every row actually added
    is logged to that chapter's pipeline_changes_<Chapter>.txt - the same
    shared log every other notebook writes to, via save_inconsistencies().
    Left as None for a dictionary-wide call with no one chapter behind it,
    which logs to the _general file instead.

    Appends to the module-level DICTIONARY_LOG rather than logging just this
    call's own rows - a run that calls this twice (once for Kind 1 gaps,
    once for Kind 3) used to have the second call's save_inconsistencies()
    silently replace the first call's DICTIONARY section, since a section is
    always rewritten in full from whatever records it is given. Passing the
    whole accumulated log every time means neither call's rows are lost.
    """
    if not isinstance(filled, pd.DataFrame):
        filled = pd.read_excel(filled, engine="openpyxl")

    needed = ["col_ar", "val_ar", "col_en", "val_en"]
    missing = [c for c in needed if c not in filled.columns]
    if missing:
        raise ValueError(f"missing column(s) {missing}; expected {needed}")

    candidates = filled[needed].copy()
    is_column_row = candidates["val_ar"].isna() & candidates["val_en"].isna()
    column_rows = candidates[is_column_row
                             & candidates["col_ar"].notna() & candidates["col_en"].notna()]

    value_rows = candidates[~is_column_row].dropna()
    value_rows = value_rows[(value_rows["val_ar"].astype(str).str.strip() != "")
                            & (value_rows["val_en"].astype(str).str.strip() != "")]

    new_rows = pd.concat([column_rows, value_rows], ignore_index=True)
    if new_rows.empty:
        logger.warning("No completed rows to add - is val_en filled in?")
        return None

    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    already_there = {dictionary_key(c, v)
                     for c, v in zip(dictionary["col_ar"], dictionary["val_ar"])}
    to_add = new_rows[~new_rows.apply(
        lambda r: dictionary_key(r["col_ar"], r["val_ar"]) in already_there, axis=1)]
    if to_add.empty:
        logger.info("Every row is already in the dictionary - nothing to add.")
        return dictionary

    if backup:
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        backup_path = TRANSLATION_DICT_PATH.with_name(
            f"{TRANSLATION_DICT_PATH.stem} backup {stamp}.xlsx")
        dictionary.to_excel(backup_path, index=False, engine="openpyxl")
        logger.info(f"Backed up the dictionary to {backup_path.name}")

    # Mark what the pipeline added, so a row typed by hand and a row translated
    # by Claude Code can be told apart later. Originals keep a blank status.
    to_add = to_add.reindex(columns=dictionary.columns)
    if "status" in to_add.columns:
        to_add["status"] = "updated"

    updated = pd.concat([dictionary, to_add], ignore_index=True)
    updated.to_excel(TRANSLATION_DICT_PATH, index=False, engine="openpyxl")
    logger.info(f"Added {len(to_add):,} row(s) to {TRANSLATION_DICT_PATH.name} "
                f"({len(dictionary):,} -> {len(updated):,}). Re-run to use them.")

    DICTIONARY_LOG.extend({
        "kind": "dictionary entry added", "chapter": chapter or "-",
        "detail": (f"column name added: {row['col_ar']!r} -> {row['col_en']!r}"
                  if pd.isna(row["val_ar"]) else
                  f"{row['col_en']}: {row['val_ar']!r} -> {row['val_en']!r}"),
    } for _, row in to_add.iterrows())
    save_inconsistencies("DICTIONARY", DICTIONARY_LOG, chapters=[chapter] if chapter else None)

    return updated


def calculated_labels():
    """Every label this notebook INVENTS, as (English column, English value).

    These come from the calculations, not from any questionnaire, so the
    dictionary can only ever learn them from here. Derived from the same
    constants the calculations use, so adding a calculation cannot leave this
    list behind.
    """
    labels = [("Indicator", SEX_RATIO_TITLE), ("Indicator", AGE_SHARE_TITLE)]
    labels += [("Age Group", group) for group in AGE_GROUPS]
    return labels


def check_calculated_labels():
    """Which invented labels the dictionary does not know yet.

    Returned in the gap shape used elsewhere, except that here it is the ARABIC
    side that is blank: the English is what this notebook chose, and the Arabic
    is what has to be supplied before notebook 4 can render these rows.
    """
    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    known = set(zip(dictionary["col_en"].astype(str).str.strip(),
                    dictionary["val_en"].astype(str).str.strip()))

    column_map, _ = DICTIONARY_AR_TO_EN
    english_to_arabic_column = {en: ar for ar, en in column_map.items()}

    missing = []
    for english_column, label in calculated_labels():
        if (english_column, str(label).strip()) in known:
            continue
        missing.append({
            "col_ar": english_to_arabic_column.get(english_column, english_column),
            "col_en": english_column,
            "val_ar": None,          # <- to be filled in
            "val_en": label,
            "rows": None,
        })
    return pd.DataFrame(missing)


def export_calculated_labels(file_name="new_labels_to_translate.xlsx"):
    """Writes the invented labels the dictionary does not know to their own
    file, kept separate from the questionnaire gaps because the blank column is
    the other one."""
    missing = check_calculated_labels()
    if missing.empty:
        logger.info("The dictionary knows every label the calculations invent.")
        return missing
    path = COMPENDIUM_PATH / file_name
    missing.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"Saved {path.name}: {len(missing)} invented label(s) with no Arabic. "
                f"Fill in val_ar, then call update_dictionary().")
    return missing


## `build_chapter()`

Translate the Arabic long file, append the native English one if it exists, and
save the combined result. Every step is logged, so the run output shows
exactly which chapters had an English file appended.


In [ ]:
"""
CELL: build_chapter() - translate, append the native English file, save.
"""


def build_chapter(chapter):
    """Produces COMPENDIUM-ARAB SOCIETY/<chapter>_EN.xlsx and reports what went
    into it.

    Returns a dict describing the run, so the cell below can print one table
    showing which chapters had an English long file appended and which did not.
    """
    arabic_path = LONG_FILES_PATH / f"{chapter}_AR.xlsx"
    english_path = LONG_FILES_PATH / f"{chapter}_EN_questionnaires.xlsx"
    output_path = LONG_FILES_PATH / f"{chapter}_EN.xlsx"

    report = {"chapter": chapter, "translated_rows": 0, "appended_rows": 0,
              "total_rows": 0, "appended_from": None, "untranslated": []}

    parts = []

    if arabic_path.exists():
        arabic_table = pd.read_excel(arabic_path, engine="openpyxl")
        translated = translate(arabic_table)
        parts.append(translated)
        report["translated_rows"] = len(translated)
        logger.info(f"  {chapter}: translated {len(translated):,} row(s) from "
                    f"merged_long_files\\{arabic_path.name}")
        report["untranslated"] = find_untranslated(arabic_table, translated)
    else:
        logger.info(f"  {chapter}: no Arabic long file")

    if english_path.exists():
        english_table = pd.read_excel(english_path, engine="openpyxl")
        parts.append(english_table)
        report["appended_rows"] = len(english_table)
        report["appended_from"] = f"merged_long_files\\{english_path.name}"
        logger.info(f"  {chapter}: APPENDED {len(english_table):,} row(s) from "
                    f"{report['appended_from']} (already English, not translated)")
    else:
        logger.info(f"  {chapter}: no English long file to append")

    if not parts:
        logger.warning(f"  {chapter}: nothing to build - run notebook 1 first")
        return report

    # Concatenate, not merge side-by-side: both parts are the same long shape,
    # so this stacks the English-sourced rows under the translated ones.
    combined = pd.concat(parts, ignore_index=True)
    combined.to_excel(output_path, index=False, engine="openpyxl")
    report["total_rows"] = len(combined)
    logger.info(f"  {chapter}: saved {output_path.name} ({len(combined):,} rows)")
    return report


## Run - translate, append, save


In [ ]:
"""
CELL: Main run - build every chapter's combined English file.
"""
print(f"Translating {SOURCE_LANGUAGE} -> {TARGET_LANGUAGE}")
print(f"  reading   {LONG_FILES_PATH}")
print(f"  appending <Chapter>_EN_questionnaires.xlsx, where one exists")
print(f"  writing   {LONG_FILES_PATH}\\<Chapter>_EN.xlsx\n")

REPORTS = []
chapters = chapters_to_process()
total_chapters = len(chapters)
for i, chapter in enumerate(chapters, start=1):
    bar = "#" * i + "-" * (total_chapters - i)
    print(f"[{bar}] chapter {i}/{total_chapters}: {chapter}")
    REPORTS.append(build_chapter(chapter))

print("\n" + "=" * 78)
print("WHAT WENT INTO EACH FILE")
print("=" * 78)
print(f"{'Chapter':<12}{'translated AR':>15}{'appended EN':>14}{'total':>12}   appended from")
for r in REPORTS:
    appended = r["appended_from"] or "-  (no English long file)"
    print(f"{r['chapter']:<12}{r['translated_rows']:>15,}{r['appended_rows']:>14,}"
          f"{r['total_rows']:>12,}   {appended}")

with_english = [r["chapter"] for r in REPORTS if r["appended_rows"]]
print(f"\nChapters that had an English long file appended: "
      f"{with_english if with_english else 'none'}")

gap_count = len({(g["col_ar"], g["val_ar"]) for r in REPORTS for g in r["untranslated"]})
print(f"Distinct values with no dictionary entry: {gap_count}")
if gap_count:
    print("They are listed in each chapter's own pipeline_inconsistencies_<Chapter>.txt,")
    print("with the country, indicator and year they appear in. Ask Claude Code")
    print("to fill the dictionary gaps, then run this cell again.")

# ------------------------------------------------------- the shared log
records = gap_records(REPORTS)
paths, count = save_inconsistencies("4. TRANSLATION", records, chapters=chapters)
print(f"{count} value(s) with no translation recorded across {len(paths)} file(s)")


## Gaps

Written to the shared log with the country, indicator and year they
appear in. No spreadsheet is produced - the log is the record.


In [ ]:
"""
CELL: The gaps found above, as a table to read here.
"""
pd.DataFrame(gap_records(REPORTS))


## Folding in external data

Notebook 3 extracts published tables that never went through a questionnaire
and writes each chapter's rows, already in English, to
`<Chapter>_EN_external.xlsx` - it does none of the translation itself, so
this notebook is the one place responsible for every value's language,
whichever direction it travels. This section reads that file, appends its
rows to `<Chapter>_EN.xlsx` (freshly rebuilt above, so there is nothing of
its own to strip first), and gives each value an Arabic form via the
dictionary - Claude, reviewing the run, supplies whatever the dictionary
does not already have and `update_dictionary()` records it, the same
"Filling dictionary gaps" loop `../CLAUDE.md` describes, just travelling
the other way: this is genuinely new English-origin content getting an
Arabic form for the first time, not a back-translation of anything already
correct - see `translation dict.xlsx`'s own rule there.

Every row this appends carries `Data Origin` = `External`
(`مصدر البيانات` = `بيانات خارجية` on the Arabic side) so tabulations can
keep ignoring it - see notebook 3's own intro for why.

Run **Fold in external data - part 1** below, fill in any `EXTERNAL_GAPS`,
call `update_dictionary(filled, chapter="<Chapter>")`, then
**`apply_external_gaps()`**.
Idempotent on the Arabic side: re-running strips this section's own
previously-added rows first, by `Data Origin`, so nothing duplicates. The
English side needs no such trick - `run` above already rebuilt
`<Chapter>_EN.xlsx` from scratch this run, so appending to it here can
never duplicate anything.

In [ ]:
"""
CELL: The dictionary, read English to Arabic too - for genuinely new
English-origin content from notebook 3 only, never a back-translation of
anything already correct. See the intro above and translation dict.xlsx's
own rule in ../CLAUDE.md.
"""

# Same number and same meaning as the rest of the pipeline: a value just
# above this line is trustworthy enough for the real pipeline to fix on its
# own without review.
FUZZY_MATCH_CUTOFF = 0.6

# Never fuzzy-matched, only ever exact - a long, templated Indicator or
# Source sentence carries its topic in a small fraction of the string, so
# two different ones sharing a structure can still score above the cutoff.
# Measured, not guessed: this project's own health-spending text scored
# 0.628 against its education-spending text on nothing but shared wording.
NEVER_FUZZY_MATCHED = {"Source", "Indicator"}

ORIGIN_COLUMN_EN = "Data Origin"
ORIGIN_COLUMN_AR = "مصدر البيانات"
ORIGIN_EXTERNAL_EN = "External"
ORIGIN_EXTERNAL_AR = "بيانات خارجية"

# Year and Value never need a value translated - a year is a year, a number
# is a number - but the COLUMN NAME still has to match notebook 1's own
# constants of the same name, or notebook 5's tabulations cannot find them:
# they look for "السنة" and "العدد" by name, not by position.
YEAR_COLUMN_AR = "السنة"
VALUE_COLUMN_AR = "العدد"

# Categorical columns that need an Arabic form.
TRANSLATABLE_COLUMNS = ["Country", "Indicator", "Sex", "Source"]


def load_english_to_arabic():
    """The dictionary read the other way - English value -> Arabic value,
    per column - built from the SAME translation dict.xlsx the AR->EN
    direction above already loaded, just indexed the other way."""
    dict_df = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    english_column_map = {}
    english_value_map = {}
    for english_column in dict_df["col_en"].dropna().unique():
        rows = dict_df[dict_df["col_en"] == english_column]
        english_column_map[english_column] = (
            rows["col_ar"].iloc[0] if rows["col_ar"].notna().any() else None)
        english_value_map[english_column] = {
            str(en).strip(): ar
            for en, ar in zip(rows["val_en"], rows["val_ar"])
            if pd.notna(en) and pd.notna(ar)
        }
    return english_column_map, english_value_map


ENGLISH_COLUMN_MAP, ENGLISH_VALUE_MAP = load_english_to_arabic()


def arabic_for_column(english_column):
    """The Arabic name for an English column - added to the running map
    (not the file - only update_dictionary() does that) the first time this
    notebook itself needs one, so a second lookup in the same run reuses it."""
    return ENGLISH_COLUMN_MAP.get(english_column)


def arabic_for_value(english_column, english_value):
    """The Arabic value already on file for this exact (column, value) pair,
    or None if nothing has ever recorded one."""
    return ENGLISH_VALUE_MAP.get(english_column, {}).get(str(english_value).strip())


def best_match(text, choices):
    """The known choice difflib considers closest to text, and how close (0
    to 1) - identical to notebook 1's function of the same name."""
    best_choice, best_score = None, -1
    for choice in choices:
        score = difflib.SequenceMatcher(None, str(text), str(choice)).ratio()
        if score > best_score:
            best_choice, best_score = choice, score
    return best_choice, best_score


FUZZY_MATCHES = []   # every value resolved by a close-enough match, not an exact one


def resolve_arabic(english_column, english_value):
    """The one place every gap-or-not decision for external data is made, so
    find_gaps() and translate_to_arabic() can never disagree with each other.

    Returns (val_ar, method): "exact" for a value already on file under this
    exact spelling, "fuzzy" for a close match reused from a DIFFERENT
    spelling already on file (a country's long form against its short form -
    "State of Palestine" against "Palestine" - reusing its Arabic rather than
    filing a second, inconsistent entry), or (None, None) if nothing on file
    is close enough and this is a real gap.

    NEVER_FUZZY_MATCHED columns skip straight from exact to "no match, real
    gap" - proven necessary empirically: "Government expenditure on health as
    % of Gross Domestic Product (GDP)" scored 0.628 against the dictionary's
    *education* spending indicator, over the cutoff, on nothing but shared
    sentence template.
    """
    exact = arabic_for_value(english_column, english_value)
    if exact is not None:
        return exact, "exact"
    if english_column in NEVER_FUZZY_MATCHED:
        return None, None

    known = ENGLISH_VALUE_MAP.get(english_column, {})
    if not known:
        return None, None
    match, score = best_match(str(english_value).strip(), known.keys())
    if match is not None and score >= FUZZY_MATCH_CUTOFF:
        return known[match], "fuzzy"
    return None, None


In [ ]:
"""
CELL: Appending external data - to English directly, to Arabic once the
dictionary has an entry for everything new.
"""


def strip_previous_external(table, chapter):
    """Remove this chapter's own previously-appended external rows before
    adding fresh ones - the idempotent pattern notebook 2 uses for its
    calculated rows, applied here to the Arabic side only. The English side
    needs no such trick: run_translation() above already rebuilt
    <Chapter>_EN.xlsx from scratch this run, so it starts external-free
    every time."""
    if ORIGIN_COLUMN_AR not in table.columns:
        return table
    is_previous = table[ORIGIN_COLUMN_AR] == ORIGIN_EXTERNAL_AR
    if "Chapter" in table.columns:
        is_previous &= table["Chapter"].astype(str).str.strip() == chapter
    removed = int(is_previous.sum())
    if removed:
        logger.info(f"  {chapter}: removing {removed:,} external row(s) from a previous run")
    return table[~is_previous]


def append_external_to_english(chapter, new_rows):
    """Appends new_rows to the <Chapter>_EN.xlsx the main run above just
    rebuilt. No strip needed first - that file starts external-free every
    run."""
    path = LONG_FILES_PATH / f"{chapter}_EN.xlsx"
    existing = pd.read_excel(path, engine="openpyxl") if path.exists() else pd.DataFrame()
    combined = pd.concat([existing, new_rows], ignore_index=True)
    combined.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"  {chapter}: {len(new_rows):,} external row(s) -> {path.name} "
               f"({len(combined):,} rows)")
    return combined


def find_external_gaps(new_rows):
    """Every (column, value) notebook 3's output needs an Arabic form for and
    does not already have one - the chapter name included, since that needs
    translating exactly once too.

    A close-enough fuzzy match against an existing English spelling is
    reused rather than filed as a gap, but is always reported in
    FUZZY_MATCHES - a good score is not certainty, and a wrong reuse would
    quietly merge two different things into one row everywhere downstream.
    """
    gaps, fuzzy = [], []
    seen = set()

    def check(col_en, val_en):
        val_en = str(val_en).strip()
        key = (col_en, val_en)
        if key in seen or val_en == "" or val_en.lower() == "nan":
            return
        seen.add(key)
        resolved, method = resolve_arabic(col_en, val_en)
        if resolved is None:
            gaps.append({
                "col_en": col_en, "val_en": val_en,
                "col_ar": arabic_for_column(col_en),   # None if the column itself is new too
            })
        elif method == "fuzzy":
            fuzzy.append({"col_en": col_en, "val_en": val_en, "matched_val_ar": resolved})

    for column in TRANSLATABLE_COLUMNS:
        if column not in new_rows.columns:
            continue
        for value in new_rows[column].dropna().unique():
            check(column, value)

    for chapter in new_rows.get("Chapter", pd.Series(dtype=str)).dropna().unique():
        check("Chapter", chapter)

    FUZZY_MATCHES.extend(fuzzy)
    return pd.DataFrame(gaps, columns=["col_en", "val_en", "col_ar"])


def translate_external_to_arabic(table):
    """The external English rows notebook 3 wrote, rebuilt with Arabic column
    names and values - only possible once every value find_external_gaps()
    reported has a dictionary entry, which update_dictionary() supplies."""
    global ENGLISH_COLUMN_MAP, ENGLISH_VALUE_MAP
    ENGLISH_COLUMN_MAP, ENGLISH_VALUE_MAP = load_english_to_arabic()   # pick up what update_dictionary() just added

    result = table.copy()
    still_missing = []
    for column in TRANSLATABLE_COLUMNS + ["Chapter"]:
        if column not in result.columns:
            continue
        arabic_column = arabic_for_column(column)
        if arabic_column is None:
            still_missing.append(column)
            continue

        # Resolve each DISTINCT value once, not once per row - resolve_arabic()
        # can fall back to a fuzzy scan of every known value in the column, and
        # doing that per row rather than per distinct value turns a few hundred
        # lookups into several million on a several-thousand-row sheet.
        lookup = {v: resolve_arabic(column, v)[0]
                 for v in result[column].dropna().unique()}
        mapped = result[column].map(lookup)
        unresolved = result[column].notna() & mapped.isna()
        if unresolved.any():
            still_missing.extend(
                f"{column}: {v!r}" for v in result.loc[unresolved, column].unique())
        result[column] = mapped
        result = result.rename(columns={column: arabic_column})

    if ORIGIN_COLUMN_EN in result.columns:
        result[ORIGIN_COLUMN_EN] = result[ORIGIN_COLUMN_EN].map(
            {ORIGIN_EXTERNAL_EN: ORIGIN_EXTERNAL_AR}).fillna(result[ORIGIN_COLUMN_EN])
        result = result.rename(columns={ORIGIN_COLUMN_EN: ORIGIN_COLUMN_AR})

    result = result.rename(columns={"Year": YEAR_COLUMN_AR, "Value": VALUE_COLUMN_AR})

    if still_missing:
        raise ValueError(
            "still missing an Arabic form for: " + ", ".join(map(str, still_missing[:10]))
            + " - call find_external_gaps() again and update_dictionary() before retrying")
    return result


def append_external_to_arabic(chapter, ar_rows):
    """The Arabic twin of append_external_to_english() - strips this
    section's own previous contribution first, since <Chapter>_AR.xlsx is
    not rebuilt from scratch the way <Chapter>_EN.xlsx is."""
    path = LONG_FILES_PATH / f"{chapter}_AR.xlsx"
    existing = pd.read_excel(path, engine="openpyxl") if path.exists() else pd.DataFrame()
    existing = strip_previous_external(existing, chapter)

    combined = pd.concat([existing, ar_rows], ignore_index=True)
    combined.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"  {chapter}: {len(ar_rows):,} external row(s) -> {path.name} "
               f"({len(combined):,} rows)")
    return combined


In [ ]:
"""
CELL: Fold in external data - part 1. Reads every chapter's
<Chapter>_EN_external.xlsx (written by notebook 3, if it ran), appends to
English, and collects the translation gaps. Stop and review before calling
apply_external_gaps() below - nothing is written to Arabic, or to the
dictionary, until that is called.
"""
FUZZY_MATCHES.clear()

external_files = {
    chapter: LONG_FILES_PATH / f"{chapter}_EN_external.xlsx"
    for chapter in chapters_to_process()
}
external_files = {c: p for c, p in external_files.items() if p.exists()}
print(f"Chapters with external data to fold in: {list(external_files)}\n")

APPENDED_EXTERNAL = {}   # chapter -> the rows this run appended (needed by apply_external_gaps())
ALL_EXTERNAL_GAPS = []

for chapter, path in external_files.items():
    print(f"=== {chapter} ===")
    new_rows = pd.read_excel(path, engine="openpyxl")
    if new_rows.empty:
        print("  nothing usable found\n")
        continue

    append_external_to_english(chapter, new_rows)
    APPENDED_EXTERNAL[chapter] = new_rows

    gaps = find_external_gaps(new_rows)
    if not gaps.empty:
        gaps.insert(0, "chapter", chapter)
        ALL_EXTERNAL_GAPS.append(gaps)
    print()

EXTERNAL_GAPS = pd.concat(ALL_EXTERNAL_GAPS, ignore_index=True) if ALL_EXTERNAL_GAPS else pd.DataFrame(
    columns=["chapter", "col_en", "val_en", "col_ar"])

print("=" * 70)
print(f"{sum(len(v) for v in APPENDED_EXTERNAL.values()):,} external row(s) appended across "
     f"{len(APPENDED_EXTERNAL)} chapter(s)")
print(f"{len(EXTERNAL_GAPS)} value(s) need an Arabic translation before appending to "
     f"the Arabic files - see EXTERNAL_GAPS")
print(f"{len(FUZZY_MATCHES)} value(s) reused a close-enough existing Arabic "
     f"spelling instead - worth a glance, see FUZZY_MATCHES")

if FUZZY_MATCHES:
    print("\nFuzzy matches - reused, not filed as gaps:")
    for row in FUZZY_MATCHES:
        print(f"  [{row['col_en']}] {row['val_en']!r} -> {row['matched_val_ar']!r}")

records = [{"kind": "needs Arabic translation", "chapter": row["chapter"],
            "detail": f"[{row['col_en']}] {row['val_en']!r} has no Arabic form on file yet"}
           for _, row in EXTERNAL_GAPS.iterrows()]
records += [{"kind": "reused a close Arabic spelling",
            "detail": f"[{row['col_en']}] {row['val_en']!r} -> {row['matched_val_ar']!r}"}
           for row in FUZZY_MATCHES]
paths, count = save_inconsistencies("4b. EXTERNAL DATA", records, chapters=list(external_files))
print(f"\n{count} inconsistency(ies) recorded across {len(paths)} file(s)")

if not EXTERNAL_GAPS.empty:
    print("\nEXTERNAL_GAPS:")
    print(EXTERNAL_GAPS.to_string(index=False))


In [ ]:
"""
CELL: Fold in external data - part 2. Call once EXTERNAL_GAPS has been
filled in and update_dictionary() has been called. Builds the Arabic twin
of every row appended above and writes it to <Chapter>_AR.xlsx.

    filled = EXTERNAL_GAPS.copy()
    filled["val_ar"] = [...]            # by position, never by retyping val_en
    update_dictionary(filled, chapter="<Chapter>")
    apply_external_gaps()
"""


def apply_external_gaps():
    if not APPENDED_EXTERNAL:
        logger.warning("Nothing was appended this run - nothing to translate to Arabic.")
        return

    for chapter, new_rows in APPENDED_EXTERNAL.items():
        ar_rows = translate_external_to_arabic(new_rows)
        append_external_to_arabic(chapter, ar_rows)

    print(f"\nArabic rows written for: {list(APPENDED_EXTERNAL)}")
